# 🚀 LLM-based Feature Engineering using Ollama (Fully Free)

This notebook builds a fraud detection pipeline using:
- Ollama (local LLM)
- Behavioral feature generation
- DistilBERT model
- Iterative loop (semi-automated)


## ✅ STEP 0: Install Ollama (RUN IN TERMINAL, NOT NOTEBOOK)

```bash
curl -fsSL https://ollama.com/install.sh | sh
ollama pull llama3
ollama run llama3
```

Keep Ollama running in background.

In [1]:
# STEP 1: Install dependencies
!pip install pandas numpy scikit-learn transformers datasets tqdm requests

In [2]:
# STEP 2: Load dataset
import pandas as pd

df = pd.read_csv('../../transactions/card_transaction.v1.csv')
df.head()

,User,Card,Year,Month,Day,Time,Amount,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors?,Is Fraud?
0,0,0,2002,9,1,06:21,$134.09,Swipe Transaction,3527213246127876953,La Verne,CA,91750.0,5300,NaN,No
1,0,0,2002,9,1,06:42,$38.48,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No
2,0,0,2002,9,2,06:22,$120.34,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No
3,0,0,2002,9,2,17:45,$128.95,Swipe Transaction,3414527459579106770,Monterey Park,CA,91754.0,5651,NaN,No
4,0,0,2002,9,3,06:23,$104.71,Swipe Transaction,5817218446178736267,La Verne,CA,91750.0,5912,NaN,No


In [3]:
# STEP 3: Create user-level fraud label
fraud_users = df.groupby('User')['Is Fraud?'].max().reset_index()
fraud_users.columns = ['User', 'User_Fraud_Label']

df = df.merge(fraud_users, on='User')
df.head()

,User,Card,Year,Month,Day,Time,Amount,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors?,Is Fraud?,User_Fraud_Label
0,0,0,2002,9,1,06:21,$134.09,Swipe Transaction,3527213246127876953,La Verne,CA,91750.0,5300,NaN,No,Yes
1,0,0,2002,9,1,06:42,$38.48,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No,Yes
2,0,0,2002,9,2,06:22,$120.34,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No,Yes
3,0,0,2002,9,2,17:45,$128.95,Swipe Transaction,3414527459579106770,Monterey Park,CA,91754.0,5651,NaN,No,Yes
4,0,0,2002,9,3,06:23,$104.71,Swipe Transaction,5817218446178736267,La Verne,CA,91750.0,5912,NaN,No,Yes


In [6]:
# STEP 4: Connect to Ollama
import requests

def call_llm(prompt):
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "llama3",
            "prompt": prompt,
            "stream": False
        }
    )
    return response.json()['response']

In [8]:
# STEP 5: Generate features using LLM

schema = list(df.columns)

prompt = f"""
Dataset columns: {schema}

Generate fraud detection behavioral features.

STRICT JSON OUTPUT ONLY:
[
  {{
    "feature_name": "",
    "description": "",
    "type": "numerical/categorical",
    "computation": "",
    "intuition": ""
  }}
]
"""

llm_output = call_llm(prompt)
print(llm_output)

Here are some fraud detection behavioral features that can be generated from the given dataset:

```
[
  {
    "feature_name": "Average Daily Spend",
    "description": "The average amount spent by a user per day",
    "type": "numerical",
    "computation": "SUM(Amount) / COUNT(Day)",
    "intuition": "Users who spend consistently large amounts on a daily basis may be more likely to be fraudulent"
  },
  {
    "feature_name": "Number of Transactions in a Month",
    "description": "The number of transactions made by a user within a month",
    "type": "numerical",
    "computation": "COUNT(Transaction Date)",
    "intuition": "Users who make a large number of transactions in a short period may be more likely to be fraudulent"
  },
  {
    "feature_name": "Percentage of Transactions with Chip Use",
    "description": "The percentage of transactions where the chip was used",
    "type": "numerical",
    "computation": "COUNT(Use Chip = 'True') / COUNT(*) * 100",
    "intuition": "Users 

In [ ]:
# STEP 6: Example feature implementation (manual first iteration)

print(df['Amount'].dtype)
print(df['Amount'].head(10))
df['Amount'] = df['Amount'].replace('[\$,]', '', regex=True).astype(float)

df['txn_count'] = df.groupby('User')['Amount'].transform('count')
df['avg_amount'] = df.groupby('User')['Amount'].transform('mean')
df['amount_std'] = df.groupby('User')['Amount'].transform('std')
df['unique_merchants'] = df.groupby('User')['Merchant Name'].transform('nunique')

df.head()

object
0    $134.09
1     $38.48
2    $120.34
3    $128.95
4    $104.71
5     $86.19
6     $93.84
7    $123.50
8     $61.72
9     $57.10
Name: Amount, dtype: object


TypeError: agg function failed [how->mean,dtype->object]

In [ ]:
# STEP 7: Convert features to text for DistilBERT

def row_to_text(row):
    return f"User behavior: avg {row['avg_amount']}, count {row['txn_count']}, std {row['amount_std']}"

df['text'] = df.apply(row_to_text, axis=1)
df[['text', 'User_Fraud_Label']].head()

In [ ]:
# STEP 8: Train DistilBERT
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

dataset = Dataset.from_pandas(df[['text', 'User_Fraud_Label']])

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

def tokenize(example):
    return tokenizer(example['text'], truncation=True, padding='max_length')

dataset = dataset.map(tokenize, batched=True)
dataset = dataset.train_test_split(test_size=0.2)

model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    evaluation_strategy='epoch'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test']
)

trainer.train()

In [ ]:
# STEP 9: Evaluate
metrics = trainer.evaluate()
print(metrics)

## 🔁 STEP 10: Iteration (Manual + Guided)

If performance is low:
- Re-run STEP 5 with improved prompt
- Add/remove features
- Retrain model

Optional: Add feedback into prompt like:

Model struggling with temporal patterns → generate time-based features
